<a href="https://colab.research.google.com/github/louisnguyen-eep/AgenticAIforBusiness118S/blob/dev/refundPoliciesAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install anthropic langgraph langchain-core -q

import re
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

import anthropic
from google.colab import userdata

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# ── Claude Client ─────────────────────────────────────────────────────────────
api_key = userdata.get('ANTHROPIC_API_KEY')
client  = anthropic.Anthropic(api_key=api_key)
MODEL   = "claude-sonnet-4-5"

# ── Refund Policy ─────────────────────────────────────────────────────────────
REFUND_POLICY = {
    "window":        "30 days from the delivery date",
    "condition":     "Items must be unused, in original packaging, with all accessories included",
    "process_time":  "3–5 business days after the returned item is received",
    "method":        "Refund issued to the original payment method only",
    "exceptions":    "Opened software licenses and gift cards are non-refundable",
    "damaged_items": "If your item arrived damaged or defective, we cover return shipping and offer a full refund or replacement",
    "wrong_item":    "If you received the wrong item, contact us within 7 days and we will arrange a free return and re-ship",
    "how_to_start":  "Visit nexastore.com/returns, enter your order number, and follow the steps to print a return label",
    "contact":       "support@nexastore.com | Monday–Friday, 9 AM – 6 PM EST",
}

# ── Refund Cases Database ─────────────────────────────────────────────────────
REFUND_CASES = {
    "REF-2001": {"status": "Approved",    "order": "ORD-1003", "item": "SoundDrop ANC",   "amount": "$249",  "issued_date": "March 27, 2026"},
    "REF-2002": {"status": "Pending",     "order": "ORD-1008", "item": "NexaPad Lite",    "amount": "$449",  "issued_date": "Awaiting item return"},
    "REF-2003": {"status": "Rejected",    "order": "ORD-1001", "item": "NexaPad Ultra",   "amount": "$0",    "issued_date": "N/A — outside return window"},
    "REF-2004": {"status": "In Progress", "order": "ORD-1009", "item": "SoundDrop Go",    "amount": "$99",   "issued_date": "Processing, 2–3 days remaining"},
    "REF-2005": {"status": "Approved",    "order": "ORD-1005", "item": "NexaCam 4K",      "amount": "$349",  "issued_date": "March 30, 2026"},
    "REF-2006": {"status": "Pending",     "order": "ORD-1007", "item": "DeskHub Pro",     "amount": "$179",  "issued_date": "Awaiting item return"},
}

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are Alex, a friendly customer support agent for NexaStore, a premium online tech retailer.
Your only job in this session is to help customers with refund requests and return policy questions.

BEHAVIOUR RULES:
- Be warm, empathetic, concise, and professional.
- Always explain the relevant policy clearly before asking for more details.
- Use the exact refund case details provided in SYSTEM NOTEs — never invent information.
- If a customer asks about a specific refund case (REF-XXXX), use those details.
- If no refund case number is provided but they want to start a return, guide them to nexastore.com/returns.
- For damaged or wrong items, express extra empathy and assure them we will make it right.

COMPANY DETAILS:
- Support email : support@nexastore.com
- Support hours : Monday–Friday, 9 AM – 6 PM EST
- Return policy : 30-day hassle-free returns
- Website       : nexastore.com
""".strip()

# ── Claude Intent Classifier ──────────────────────────────────────────────────
def detect_intent(user_message: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        system="""Classify the customer message into exactly one of these intents:
check_case, damaged, wrong_item, how_to, policy, general

Reply with only the intent label, nothing else.""",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip().lower()

# ── Refund Case Lookup ────────────────────────────────────────────────────────
def lookup_refund_case(message: str) -> str | None:
    match = re.search(r'REF-\d+', message, re.IGNORECASE)
    if not match:
        return None
    case_id = match.group().upper()
    if case_id not in REFUND_CASES:
        return f"[SYSTEM NOTE: Refund case {case_id} was not found. Tell the customer and ask them to double-check.]"
    r = REFUND_CASES[case_id]
    return (
        f"[SYSTEM NOTE - Refund case details for {case_id}:\n"
        f"  Order: {r['order']} | Item: {r['item']} | Status: {r['status']} | "
        f"Amount: {r['amount']} | Issued/Expected: {r['issued_date']}\n"
        f"Use these exact details in your reply.]"
    )

# ── Policy Context Builder ────────────────────────────────────────────────────
def build_policy_context(intent: str) -> str:
    p = REFUND_POLICY
    if intent == "damaged":
        note = p["damaged_items"]
    elif intent == "wrong_item":
        note = p["wrong_item"]
    elif intent == "how_to":
        note = f"How to start a return: {p['how_to_start']}"
    else:
        note = (
            f"Return window: {p['window']} | "
            f"Condition: {p['condition']} | "
            f"Refund issued in: {p['process_time']} | "
            f"Method: {p['method']} | "
            f"Exceptions: {p['exceptions']} | "
            f"How to start: {p['how_to_start']}"
        )
    return f"[SYSTEM NOTE - Relevant refund policy:\n  {note}\n  Contact: {p['contact']}\nUse this in your reply.]"

# ── LangGraph State ───────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ── Claude Node ───────────────────────────────────────────────────────────────
def claude_node(state: AgentState) -> dict:
    claude_messages = []
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            claude_messages.append({"role": "user",      "content": msg.content})
        elif isinstance(msg, AIMessage):
            claude_messages.append({"role": "assistant", "content": msg.content})

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=claude_messages,
    )
    reply = response.content[0].text.strip()
    return {"messages": [AIMessage(content=reply)]}

# ── Build Graph ───────────────────────────────────────────────────────────────
def build_graph() -> StateGraph:
    memory  = MemorySaver()
    builder = StateGraph(AgentState)
    builder.add_node("alex", claude_node)
    builder.add_edge(START, "alex")
    builder.add_edge("alex", END)
    return builder.compile(checkpointer=memory)

GRAPH = build_graph()

def invoke_graph(thread_id: str, human_content: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = GRAPH.invoke(
        {"messages": [HumanMessage(content=human_content)]},
        config=config,
    )
    return result["messages"][-1].content

# ── Main Chat Session ─────────────────────────────────────────────────────────
def run_chat_session():
    thread_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("=" * 60)
    print("  NexaStore — Refund & Returns Agent")
    print(f"  Session ID (MemorySaver thread): {thread_id}")
    print("=" * 60)
    print("  Available refund cases: REF-2001 through REF-2006")
    print("  Type your message and press Enter. Type 'done' to exit.")
    print("-" * 60)

    # Greeting
    greeting = invoke_graph(thread_id, "Greet the customer warmly and let them know you can help with refunds and returns.")
    print(f"\n  Alex: {greeting}\n")

    # Conversation loop
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("done", "quit", "exit", "bye"):
            break

        intent = detect_intent(user_input)
        print(f"  [Intent: {intent}]")

        case_note = lookup_refund_case(user_input)
        if case_note:
            augmented = f"{user_input}\n\n{case_note}"
        else:
            policy_note = build_policy_context(intent)
            augmented   = f"{user_input}\n\n{policy_note}"

        reply = invoke_graph(thread_id, augmented)
        print(f"\n  Alex: {reply}\n")
        print("-" * 60)

    # Closing
    closing = invoke_graph(thread_id, "The customer is leaving. Give a warm one-sentence goodbye.")
    print(f"\n  Alex: {closing}\n")
    print("=" * 60)

    snapshot  = GRAPH.get_state({"configurable": {"thread_id": thread_id}})
    msg_count = len(snapshot.values["messages"])
    print(f"\n  [MemorySaver] {msg_count} messages checkpointed for thread '{thread_id}'")

# ── Run ───────────────────────────────────────────────────────────────────────
run_chat_session()

  NexaStore — Refund & Returns Agent
  Session ID (MemorySaver thread): 20260330_003739
  Type your message and press Enter. Type 'done' to exit.
------------------------------------------------------------

  Alex: Hello! Welcome to NexaStore support! 👋

I'm Alex, and I'm here to help you with any refund requests or questions about our return policy. Whether you've received a damaged item, need to return a product, or have questions about an existing refund case, I've got you covered.

How can I assist you today?

You: return
  [Intent: check_case]

  Alex: I'd be happy to help you with a return! Let me explain our return policy so you know what to expect:

**NexaStore Return Policy:**
- **Return window:** 30 days from your delivery date
- **Condition requirements:** Items must be unused, in original packaging, with all accessories included
- **Refund timeline:** 3–5 business days after we receive your returned item
- **Refund method:** Issued to your original payment method

**Please